# DCA Buy-the-Bear v5 — Progressive Reinvestment Parameter Optimisation

Grid search over start_pct, increment, and cap to find the best progressive reinvestment strategy.

In [ ]:
import pandas as pd
import requests
import time
import json
import numpy as np
from datetime import datetime, timezone
import itertools

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 220)
pd.set_option('display.float_format', lambda x: '%.4f' % x)

In [ ]:
# Fetch data once

def fetch_monthly_klines(symbol='BTCUSDT', limit=500):
    url = 'https://api.binance.com/api/v3/klines'
    all_klines = []
    start_time = 0
    while True:
        params = {'symbol': symbol, 'interval': '1M', 'startTime': start_time, 'limit': limit}
        resp = requests.get(url, params=params)
        data = resp.json()
        if not data or len(data) == 0:
            break
        all_klines.extend(data)
        print(f'Fetched {len(data)} months, up to {datetime.fromtimestamp(data[-1][0]/1000, tz=timezone.utc).strftime("%Y-%m")}')
        if len(data) < limit:
            break
        start_time = data[-1][0] + 1
        time.sleep(0.1)
    return all_klines

klines = fetch_monthly_klines()

columns = ['timestamp', 'open', 'high', 'low', 'close', 'volume',
           'close_time', 'quote_vol', 'trades', 'taker_buy_base', 'taker_buy_quote', 'ignore']
df = pd.DataFrame(klines, columns=columns)
df['date'] = pd.to_datetime(df['timestamp'], unit='ms', utc=True)
for c in ['open', 'high', 'low', 'close', 'volume']:
    df[c] = df[c].astype(float)
df = df[['date', 'open', 'high', 'low', 'close', 'volume']].copy()
df = df.sort_values('date').reset_index(drop=True)

In [ ]:
# Run one backtest for given progressive parameters

def run_backtest(start_pct, increment, cap):
    btc_held = 0.0
    usdt_reserve = 0.0
    total_invested = 0.0
    highest_close = 0.0
    reinvest_pct = start_pct
    records = []

    for i, row in df.iterrows():
        close = row['close']

        if close < row['open']:
            extra = usdt_reserve * (reinvest_pct / 100.0) if usdt_reserve > 0 else 0.0
            total_usdt = 10.0 + extra
            btc_bought = total_usdt / close
            btc_held += btc_bought
            total_invested += 10.0
            usdt_reserve -= extra

        prev_highest = highest_close
        if close > highest_close:
            highest_close = close

        if btc_held > 0.000001 and close > prev_highest:
            sell_btc = btc_held * 0.7
            sell_usdt = sell_btc * close
            btc_held -= sell_btc
            usdt_reserve += sell_usdt
            reinvest_pct = start_pct
        else:
            if reinvest_pct < cap:
                reinvest_pct = min(cap, reinvest_pct + increment)

        portfolio_value = btc_held * close + usdt_reserve
        records.append(portfolio_value)

    final_val = records[-1]
    ret_pct = (final_val / total_invested - 1) * 100

    # Max drawdown
    eq = pd.Series(records)
    running_max = eq.cummax()
    dd = (running_max - eq) / running_max
    max_dd = dd.max() * 100

    # Sharpe (annualized, skip first 12 months warmup)
    monthly_returns = eq.pct_change().dropna()
    monthly_returns = monthly_returns[monthly_returns.index >= 12]
    sharpe = (monthly_returns.mean() / monthly_returns.std()) * np.sqrt(12) if monthly_returns.std() > 0 else 0.0

    # Profit factor
    pos_sum = monthly_returns[monthly_returns > 0].sum()
    neg_sum = monthly_returns[monthly_returns < 0].abs().sum()
    pf = pos_sum / neg_sum if neg_sum > 0 else float('inf')

    # Calmar ratio
    annualized_ret = (1 + ret_pct / 100) ** (12 / len(df)) - 1
    calmar = annualized_ret / (max_dd / 100) if max_dd > 0 else 0

    return {
        'start': start_pct,
        'inc': increment,
        'cap': cap,
        'label': f'{start_pct}%+{increment}%->{cap}%',
        'return_pct': ret_pct,
        'max_dd': max_dd,
        'sharpe': sharpe,
        'calmar': calmar,
        'pf': pf,
        'final_val': final_val,
        'btc': btc_held,
    }

# Parameter grid
starts = [1, 3, 5, 10, 15]
increments = [1, 2, 3, 4, 5, 6, 7, 8, 9, 10]
caps = [30, 40, 50, 60, 70, 80, 90, 100]

print(f'Testing {len(starts) * len(increments) * len(caps)} combinations...')
results_list = []
for start, inc, cap in itertools.product(starts, increments, caps):
    r = run_backtest(start, inc, cap)
    results_list.append(r)

opt = pd.DataFrame(results_list)
print(f'Done. {len(opt)} results.')

In [ ]:
# Top 20 by Calmar ratio (best risk-adjusted)

top_calmar = opt.nlargest(20, 'calmar')
top_calmar[['label', 'return_pct', 'max_dd', 'sharpe', 'calmar', 'pf']]

In [ ]:
# Top 20 by Sharpe ratio

top_sharpe = opt.nlargest(20, 'sharpe')
top_sharpe[['label', 'return_pct', 'max_dd', 'sharpe', 'calmar', 'pf']]

In [ ]:
# Top 20 by return (with DD constraint)

top_return = opt.nlargest(20, 'return_pct')
top_return[['label', 'return_pct', 'max_dd', 'sharpe', 'calmar', 'pf']]

In [ ]:
# Find the best overall — maximize Calmar with DD < 30%

good = opt[(opt['max_dd'] < 30) & (opt['return_pct'] > 250)]
best = good.nlargest(10, 'calmar')
print(f'Best with DD < 30% and return > 250%:')
best[['label', 'return_pct', 'max_dd', 'sharpe', 'calmar', 'pf']]

In [ ]:
# Overall best by each metric

best_calmar = opt.loc[opt['calmar'].idxmax()]
best_sharpe = opt.loc[opt['sharpe'].idxmax()]
best_return = opt.loc[opt['return_pct'].idxmax()]

print("="*65)
print("OVERALL OPTIMUM BY EACH METRIC")
print("="*65)
print(f"By Calmar:  {best_calmar['label']:>15s}  Return={best_calmar['return_pct']:6.1f}%  DD={best_calmar['max_dd']:5.1f}%  Sharpe={best_calmar['sharpe']:.2f}  Calmar={best_calmar['calmar']:.2f}")
print(f"By Sharpe:  {best_sharpe['label']:>15s}  Return={best_sharpe['return_pct']:6.1f}%  DD={best_sharpe['max_dd']:5.1f}%  Sharpe={best_sharpe['sharpe']:.2f}  Calmar={best_sharpe['calmar']:.2f}")
print(f"By Return:  {best_return['label']:>15s}  Return={best_return['return_pct']:6.1f}%  DD={best_return['max_dd']:5.1f}%  Sharpe={best_return['sharpe']:.2f}  Calmar={best_return['calmar']:.2f}")

In [ ]:
# Compare with fixed-ratio best (1/11 from v3)

print("="*65)
print("BEST PROGRESSIVE vs BEST FIXED RATIO (1/11)")
print("="*65)

# Run 1/11 for direct comparison
r_111 = run_backtest(0, 0, 0)  # placeholder, we'll compute separately
# Recompute fixed strategy
btc_held = 0.0; usdt_reserve = 0.0; total_invested = 0.0; highest_close = 0.0
for i, row in df.iterrows():
    close = row['close']
    if close < row['open']:
        extra = usdt_reserve / 11.0 if usdt_reserve > 0 else 0.0
        total_usdt = 10.0 + extra
        btc_bought = total_usdt / close
        btc_held += btc_bought
        total_invested += 10.0
        usdt_reserve -= extra
    prev_highest = highest_close
    if close > highest_close:
        highest_close = close
    if btc_held > 0.000001 and close > prev_highest:
        sell_btc = btc_held * 0.7
        sell_usdt = sell_btc * close
        btc_held -= sell_btc
        usdt_reserve += sell_usdt

final_val = btc_held * df.iloc[-1]['close'] + usdt_reserve
ret = (final_val / total_invested - 1) * 100
print(f"Fixed 1/11:    Return={ret:.1f}%  Final=${final_val:.0f}  BTC={btc_held:.6f}  Reserve=${usdt_reserve:.0f}")

# Best progressive
bp = best_calmar
bp_r = run_backtest(bp['start'], bp['inc'], bp['cap'])
print(f"Progressive:   Return={bp_r['return_pct']:.1f}%  Final=${bp_r['final_val']:.0f}  BTC={bp_r['btc']:.6f}  DD={bp_r['max_dd']:.1f}%  Sharpe={bp_r['sharpe']:.2f}  Calmar={bp_r['calmar']:.2f}")

In [ ]:
# Chart: heatmap of start_pct vs increment for a fixed cap

import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

# Fix cap at 70 and show heatmaps for different metrics
fig, axes = plt.subplots(2, 2, figsize=(12, 10))

for idx, (metric, title) in enumerate([
    ('return_pct', 'Return (%)'),
    ('max_dd', 'Max Drawdown (%)'),
    ('sharpe', 'Sharpe Ratio'),
    ('calmar', 'Calmar Ratio')
]):
    ax = axes[idx // 2][idx % 2]
    pivot = opt[opt['cap'] == 70].pivot_table(
        index='start', columns='inc', values=metric, aggfunc='mean'
    )
    im = ax.imshow(pivot.values, cmap='RdYlGn' if metric != 'max_dd' else 'RdYlGn_r',
                   aspect='auto', interpolation='nearest')
    ax.set_xticks(range(len(pivot.columns)))
    ax.set_xticklabels([f'{int(c)}%' for c in pivot.columns])
    ax.set_yticks(range(len(pivot.index)))
    ax.set_yticklabels([f'{int(r)}%' for r in pivot.index])
    ax.set_xlabel('Increment per buy')
    ax.set_ylabel('Starting %')
    ax.set_title(f'{title} (cap=70%)')
    plt.colorbar(im, ax=ax)

plt.suptitle('Progressive Reinvestment — Parameter Heatmaps (cap=70%)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('dca-bt-buy-bot-optimisation-progressive.png', dpi=150, bbox_inches='tight')
print('Chart saved as dca-bt-buy-bot-optimisation-progressive.png')
plt.show()